# 05 — Block 0, operation by operation

Start at the block input T0 and **Op0 MEAN**, the first part of LayerNorm 1.
This notebook is a learning and numerical-analysis experiment, not a replacement
for the frozen LiteRT model or the verified q-projection baseline.

For each operation we will inspect its inputs, shapes, dtype, constants,
FLOAT32 calculation, proposed integer representation, concrete examples, and errors.
This first installment implements that workflow for **Op0 only**. We will extend it
in graph order after reviewing its results. GELU work is deferred until its turn.

## How to use

1. Place this file at `notebooks/05_block0_operation_walkthrough.ipynb` in the project.
2. Use the existing `../.venv` kernel. Save this notebook **before** running it.
3. Start a fresh kernel and run one code cell at a time. Read the explanation first.
4. Do not edit/save this file during a run: the existing run helper checks its file hash.
   Finish the recording cell before saving executed notebook outputs.
5. The pretrained checkpoint is loaded from the existing local cache only.
   If it is missing, stop and report the error rather than changing model revision.

Outputs use a unique `results/notebook_runs/<run-id>/` directory. They include the
input fixture and numerical report. Loading FLOAT32 on the host is for reference
and calibration; eventual target inference will use integer arithmetic.


## 1. Setup and begin a recorded run

We read the matching frozen baseline. The expected hash prevents accidentally using
the other W8A8 export. The run records the original notebook and shared-helper sources.


In [ ]:
from pathlib import Path
import sys
import hashlib
import json
import numpy as np

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/python/workflow_config.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside your project directory.')
sys.path.insert(0, str(ROOT / 'src/python'))
from workflow_config import MODEL_PATH, MODEL_ID, MODEL_REVISION, PROMPT, SEQUENCE_LENGTH
from litert_helpers import reference_interpreter, tensor_record
from notebook_runs import start_run, finish_run

NOTEBOOK_PATH = 'notebooks/05_block0_operation_walkthrough.ipynb'
EXPECTED_HASH = '39f33344ac937d13241dd964a99095eb7d4b948bd3245587d3502851431bc46a'
model_hash = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
assert model_hash == EXPECTED_HASH, 'Wrong model: stop and inspect MODEL_PATH.'
RUN_DIR = start_run(ROOT, NOTEBOOK_PATH,
    {'model_id': MODEL_ID, 'revision': MODEL_REVISION,
     'sequence_length': SEQUENCE_LENGTH, 'prompt': PROMPT,
     'scope': 'Op0 MEAN; experimental symmetric INT8 input/output',
     'scale_selection': 'maxabs/127 from this input only; not model calibration',
     'rounding': 'nearest, ties to even'}, inputs=[MODEL_PATH])
ref = reference_interpreter(MODEL_PATH)
details = {int(t['index']): t for t in ref.get_tensor_details()}
ops = ref._get_ops_details()
print('Model:', MODEL_PATH)
print('Notebook run:', RUN_DIR)


## 2. Inspect the actual first operation

T0 contains hidden features, not token IDs. For this batch-one export, Op0 reduces
batch and feature axes, leaving one mean for each token. We read the axes from T10
rather than assuming their values. T31 is still FLOAT32 in the frozen export.
There are no matrix weights in a mean calculation.


In [ ]:
op0 = next(op for op in ops if int(op['index']) == 0)
assert op0['op_name'] == 'MEAN'
assert list(map(int, op0['inputs'])) == [0, 10]
assert list(map(int, op0['outputs'])) == [31]
assert details[0]['dtype'] == np.float32
axes_raw = ref.get_tensor(10).reshape(-1)
rank = len(details[0]['shape'])
axes = tuple(int(a) % rank for a in axes_raw)
reduction_count = int(np.prod([details[0]['shape'][a] for a in axes]))
print('Op0:', op0['op_name'])
for tid in [0, 10, 31]:
    print(f'T{tid}:', tensor_record(details[tid]))
print('Reduction axes stored in T10:', axes_raw.tolist())
print('Values averaged per output:', reduction_count)
assert tuple(details[0]['shape']) == (1, 32, 128)
assert tuple(details[31]['shape']) == (32,)
assert reduction_count == 128


## 3. Obtain real block-0 input from the cached model

Use the first 32 tokens of the configured prompt. In evaluation mode the block input is
`token_embedding + position_embedding` (dropout is inactive). Embeddings are outside
block 0; we use them to supply meaningful input instead of random numbers.
We run the frozen TFLite block on the host to capture its preserved Op0 output.
This input is a new analysis case, not the frozen q-projection case-0 fixture.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, local_files_only=True)
embedding_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, local_files_only=True).cpu().eval()
encoded = tokenizer(PROMPT, add_special_tokens=False, return_tensors='pt')
assert encoded['input_ids'].shape[1] >= SEQUENCE_LENGTH
input_ids = encoded['input_ids'][:, :SEQUENCE_LENGTH]
position_ids = torch.arange(SEQUENCE_LENGTH).unsqueeze(0)
with torch.no_grad():
    x_tensor = embedding_model.transformer.drop(
        embedding_model.transformer.wte(input_ids)
        + embedding_model.transformer.wpe(position_ids))
x_fp32 = x_tensor.detach().cpu().numpy().astype(np.float32)
assert x_fp32.shape == tuple(details[0]['shape'])
ref.set_tensor(0, x_fp32)
ref.invoke()
mean_litert = ref.get_tensor(31).copy()

print('Prompt:', PROMPT)
print('Input shape / dtype:', x_fp32.shape, x_fp32.dtype)
print('Token IDs, first 8:', input_ids[0, :8].tolist())
print('Token pieces, first 8:', tokenizer.convert_ids_to_tokens(input_ids[0, :8].tolist()))
print('Input min/max:', float(x_fp32.min()), float(x_fp32.max()))
print('Token 0, first 8 features:', x_fp32[0, 0, :8])
print('LiteRT means, first 4 tokens:', mean_litert[:4])


## 4. Reproduce the FLOAT32 mean

For token t: `mean[t] = sum(x[0,t,0:128]) / 128`.
The example prints eight features, but the mean uses **all 128** features.
NumPy and LiteRT can use different summation orders, so small FLOAT32 differences
are reported separately from quantization error. A FLOAT64 mean is an additional
host diagnostic; it is not proposed target arithmetic.


In [ ]:
mean_numpy_fp32 = np.mean(x_fp32, axis=axes, dtype=np.float32)
mean_numpy_fp64 = np.mean(x_fp32.astype(np.float64), axis=axes)
print('Token 0 FLOAT32 sum:', float(np.sum(x_fp32[0, 0], dtype=np.float32)))
print('Divide by:', reduction_count)
print('Token 0 NumPy FLOAT32 mean:', float(mean_numpy_fp32[0]))
print('Token 0 LiteRT FLOAT32 mean:', float(mean_litert[0]))
print('Max NumPy FLOAT32 vs LiteRT difference:',
      float(np.max(np.abs(mean_numpy_fp32.astype(np.float64) - mean_litert))))


## 5. Propose INT8 input quantization — an explicit experiment choice

The original T0 has no quantization scale. For this first demonstration choose:

- `S = max(abs(x)) / 127`, `Z = 0`;
- `x_q = clip(round_to_nearest_even(x / S), -128, 127)`;
- diagnostic reconstruction: `x_hat = S * x_q`.

This single shared scale is selected from this one FLOAT32 input. It prevents clipping
in this example, but it is **not a fixed scale calibrated for unseen prompts** and
must not be presented as a deployable full-model scheme. It is also different from
T45's existing scale, which belongs to the output of LayerNorm 1.

We use the symmetric range [-127,127] for these fitted values. Wider integer types
are used before casting, so conversion cannot silently wrap.


In [ ]:
maxabs = float(np.max(np.abs(x_fp32.astype(np.float64))))
input_scale = maxabs / 127.0 if maxabs > 0 else 1.0
input_zero_point = 0
rounded_input = np.rint(x_fp32.astype(np.float64) / input_scale)
clipped_count = int(np.count_nonzero((rounded_input < -128) | (rounded_input > 127)))
x_q = np.clip(rounded_input, -128, 127).astype(np.int8)
x_reconstructed = x_q.astype(np.float64) * input_scale
print('Proposed scale S:', input_scale)
print('Proposed zero point Z:', input_zero_point)
print('Clipped input elements:', clipped_count)
print('Input max absolute reconstruction error:',
      float(np.max(np.abs(x_reconstructed - x_fp32))))
print('\nToken 0: first 8 features')
print('feature       FLOAT32     INT8   reconstructed      abs_error')
for k in range(8):
    original = float(x_fp32[0, 0, k])
    restored = float(x_reconstructed[0, 0, k])
    print(f'{k:7d} {original:13.8f} {int(x_q[0,0,k]):8d} '
          f'{restored:15.8f} {abs(original-restored):14.8f}')


## 6. Compute the mean using integer operations

Accumulate all 128 INT8 values in INT32. The bound is at most 128×128 = 16384 in
absolute value, so INT32 is ample here. Keep the same scale for this illustrative
INT8 mean: `mean_q = round_to_nearest_even(sum_q / 128)`.

The rounding helper uses only integer division, remainder, signs, and comparisons;
it handles negative values and ties explicitly. Its target equivalent can be C.

Two diagnostic comparisons separate the errors:

1. `S * sum_q / 128`: mean of reconstructed inputs, before output rounding;
2. `S * mean_q`: reconstructed integer output, after output rounding.

Diagnostic reconstruction uses host floating point. The integer mean path itself does not.
Rounding the mean back to INT8 may lose useful precision for LayerNorm; retaining
fractional bits or the integer sum is a later design option. This cell is not a
final LayerNorm implementation.


In [ ]:
def round_divide_nearest_even(numerator, denominator):
    numerator = np.asarray(numerator, dtype=np.int64)
    denominator = int(denominator)
    assert denominator > 0
    magnitude = np.abs(numerator)
    quotient = magnitude // denominator
    remainder = magnitude % denominator
    increment = ((2 * remainder > denominator)
                 | ((2 * remainder == denominator) & ((quotient & 1) != 0)))
    rounded = quotient + increment.astype(np.int64)
    return np.where(numerator < 0, -rounded, rounded)

sum_q = np.sum(x_q.astype(np.int32), axis=axes, dtype=np.int32)
mean_q_wide = round_divide_nearest_even(sum_q, reduction_count)
assert np.all((mean_q_wide >= -128) & (mean_q_wide <= 127))
mean_q = mean_q_wide.astype(np.int8)
mean_from_quantized_inputs = sum_q.astype(np.float64) * input_scale / reduction_count
mean_integer_reconstructed = mean_q.astype(np.float64) * input_scale
print('Integer sums, first 4:', sum_q[:4])
print('Integer means, first 4:', mean_q[:4])
print('\ntoken  LiteRT_FLOAT32  quant_input_mean  integer_mean_real  INT8')
for t in range(4):
    print(f'{t:5d} {float(mean_litert[t]):15.8f} '
          f'{mean_from_quantized_inputs[t]:17.8f} '
          f'{mean_integer_reconstructed[t]:18.8f} {int(mean_q[t]):5d}')


## 7. Report errors and record the evidence

Report absolute errors; relative errors are misleading when a mean is close to zero.
These measurements describe this input case and proposed scale, not general model
accuracy, RISC-V instructions, or hardware cycles. The fixture includes all token
features and integer intermediates so the calculation can be reproduced.

Finish this cell before saving executed notebook outputs. After completion, source
or output changes do not retroactively change the recorded experiment.


In [ ]:
def abs_metrics(reference, actual):
    error = np.abs(np.asarray(reference, dtype=np.float64)
                   - np.asarray(actual, dtype=np.float64))
    return {'max_abs': float(error.max()), 'mean_abs': float(error.mean())}

report = {
    'scope': 'Block0 Op0 MEAN only',
    'model_sha256': model_hash,
    'input_case': 'First 32 prompt tokens; cached pretrained embedding layers in eval mode',
    'prompt': PROMPT, 'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'op_index': 0, 'op_name': 'MEAN', 'input_tensor': 0, 'output_tensor': 31,
    'axes': list(axes), 'reduction_count': reduction_count,
    'quantization': {'scale': input_scale, 'zero_point': input_zero_point,
        'selection': 'One-input maxabs/127; illustrative, not calibrated across prompts',
        'rounding': 'nearest, ties to even', 'mean_output_scale': input_scale,
        'input_clipped_elements': clipped_count},
    'metrics': {
        'numpy_fp32_vs_litert': abs_metrics(mean_litert, mean_numpy_fp32),
        'numpy_fp64_vs_litert': abs_metrics(mean_litert, mean_numpy_fp64),
        'input_quantization': abs_metrics(x_fp32, x_reconstructed),
        'mean_input_quantization_only': abs_metrics(mean_litert, mean_from_quantized_inputs),
        'mean_integer_output_total': abs_metrics(mean_litert, mean_integer_reconstructed),
        'mean_output_rounding_only': abs_metrics(mean_from_quantized_inputs, mean_integer_reconstructed)},
}
fixture_path = RUN_DIR / 'op0_mean_case.npz'
report_path = RUN_DIR / 'op0_mean_report.json'
np.savez(fixture_path, input_ids=input_ids.numpy(), x_fp32=x_fp32,
    input_scale=np.array(input_scale), input_zero_point=np.array(input_zero_point),
    x_q=x_q, axes=np.array(axes), sum_q=sum_q, mean_q=mean_q,
    mean_litert=mean_litert, mean_numpy_fp32=mean_numpy_fp32,
    mean_numpy_fp64=mean_numpy_fp64,
    mean_from_quantized_inputs=mean_from_quantized_inputs,
    mean_integer_reconstructed=mean_integer_reconstructed)
report_path.write_text(json.dumps(report, indent=2) + '\n')
finish_run(ROOT, RUN_DIR, [fixture_path, report_path])
print(json.dumps(report['metrics'], indent=2))
print('Saved fixture:', fixture_path)
print('Saved report:', report_path)
print('Run complete:', RUN_DIR / 'run.json')


## Next checkpoint — review before proceeding

Send the outputs for Op0 metadata, token-0 feature examples, the first four means,
and the error metrics. We will assess whether rounding the mean to INT8 is appropriate
before following Op1 RESHAPE and Op2 SQUARED_DIFFERENCE in the variance calculation.
A shape-only operation needs inspection but does not introduce a new quantization scale.

Later sections will follow LayerNorm 1 → Q/K/V → attention → output projection →
residual → LayerNorm 2 → MLP/GELU → final residual. Already quantized operators will
also be inspected, including scales, zero points, bias, integer accumulation and
FLOAT32 diagnostic reconstruction. The frozen B0 remains the comparison checkpoint.
